In [2]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPoint, MultiPolygon, LineString, MultiLineString, GeometryCollection
from shapely.ops import unary_union, voronoi_diagram, nearest_points
from shapely import affinity

from scipy.ndimage import label
from scipy.spatial import Voronoi
from scipy.optimize import minimize
from skimage import measure

import rasterio
from rasterio.transform import from_origin
from rasterstats import zonal_stats

import sklearn.cluster


In [3]:
#------------------ FILE PATHS -----------------------
input_file = "data/Foundation_Data.gpkg"  # GeoPackage file with coastline edges and nodes
processing_file = "outputs/draft_coastal_protection_processing_layers.gpkg"  # GeoPackage file to store intermediate processing layers
output_file = "outputs/draft_Jamaica_coastal_protection_areas.gpkg"
raster_file = "data/flood_rasters/JamaicaJAM001RCPbaseline2010_epsg_32618_RP_100.tif"
#------------------------------------------------------

return_period = 100
inland_buffer_distance = 2  # Buffer distance (in kilometers) to expand the inital bounding boxes & coastline buffeering
flood_depth_threshold = 0.1  # Flood height threshold above which flood pixels will be considered
eps_threshold = 3 # DBSCAN epsilon value
minPts = 5 #DBSCAN minPts value
area_limit = 1_000_0000  # Area limit for group finalization (Used in SCAPE)
minGrpSize = 2  # Minimum number of polygons to make up a group (Used in SCAPE)
overlap_threshold = 0.50 #Minimum % groups can overlap to trigger merging (Used in SCAPE)
max_separation_distance = 8_000 #Max Distacne flooding can be from coastline to be included in flood area

coastline_edge_layer = gpd.read_file(input_file, layer="edges")  # Load edges as GeoDataFrame
coastline_node_layer = gpd.read_file(input_file, layer="nodes")  # Load nodes as GeoDataFrame

# coastline_edge_layer.to_file(output_file, layer="edges", driver="GPKG")  # Save edges to output file
# coastline_node_layer.to_file(output_file, layer="nodes", driver="GPKG")  # Save nodes to output file

jamaica_polygon_revised = gpd.read_file(input_file, layer="jam") 
jamaica_polygon_revised = jamaica_polygon_revised.to_crs(3448)
jamaica_polygon_revised["geometry"] = jamaica_polygon_revised.geometry.buffer(0)

jamaica_polygon_buffered = gpd.read_file(input_file, layer="jam_buffered") 
jamaica_polygon_buffereed = jamaica_polygon_revised.to_crs(3448)

full_coastline_layer = gpd.read_file(input_file, layer="Jamaica_Coastline_Layer")
full_coastline_layer = jamaica_polygon_revised.geometry.buffer(0).boundary



In [4]:
def add_Layer_to_File(data, output_file, layer_name, driver):
    data.to_file(output_file, layer=layer_name, driver=driver)

In [5]:
# jm = gpd.read_file(input_file, layer="jam") 
# jm = jm.to_crs(3448)
# jm["geometry"] = jm.geometry.buffer(0).boundary

# add_Layer_to_File(jm, processing_file, "TEST_buffered_coast", "GPKG")

In [103]:
# Adapted and modified from https://github.com/thomas-fred/jam-coastal-protection

def process_raster_to_clusters(threshold_m, eps, raster_file, minPts):
    """
    Processes a raster file to identify clusters of depth values using DBSCAN, 
    converts them to polygons, and returns a GeoDataFrame for further processing.

    Args:
        threshold_m (float): Minimum depth value to include pixels in the analysis.
        eps (float): DBSCAN epsilon parameter for cluster proximity.
        raster_file (str): File path to the input raster file.

    Returns:
        gpd.GeoDataFrame: GeoDataFrame containing minimum enclosing polygons for clusters.
    """
    # Open the raster file and read the first band (depth values)
    raster = rasterio.open(raster_file)
    depth_m = raster.read(1)
    i, j = np.indices(depth_m.shape)  # Create row (i) and column (j) indices

    # Extract the transformation and CRS from the raster
    transform = from_origin(raster.bounds.left, raster.bounds.top, raster.res[0], raster.res[1])
    original_crs = raster.crs

    # Create a DataFrame and filter out pixels below the depth threshold
    df = pd.DataFrame(data={"i": i.ravel(), "j": j.ravel(), "depth_m": depth_m.ravel()})
    df = df[df["depth_m"] > threshold_m]  # Keep only pixels above the threshold

    if not df.empty:
        # Initialize the DBSCAN clustering algorithm and fit the filtered data
        db = sklearn.cluster.DBSCAN(eps=eps, min_samples=minPts)
        X = df.loc[:, ["i", "j"]].to_numpy()  # Extract pixel coordinates
        db.fit(X)

        # Retrieve cluster labels and count the number of clusters
        labels = db.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)  # Exclude noise points (-1)

        if n_clusters >= 2:
            # Create polygons from clusters and convert to a GeoDataFrame
            gdf = create_polygons_from_clusters(labels, depth_m.shape, transform, original_crs, df)

            # Reproject GeoDataFrame to EPSG:3448 coordinate reference system
            gdf = gdf.to_crs("EPSG:3448")

            # Generate minimum enclosing polygons for each cluster
            gdf = create_min_enclosing_polygons(gdf)
            return gdf


def create_polygons_from_clusters(labels, shape, transform, crs, df):
    """
    Converts DBSCAN cluster labels into polygons and organizes them in a GeoDataFrame.

    Args:
        labels (np.ndarray): Cluster labels assigned by DBSCAN (-1 for noise).
        shape (tuple): Shape of the raster (rows, cols).
        transform (Affine): Transformation to convert pixel indices to coordinates.
        crs (str): Coordinate reference system of the raster.
        df (pd.DataFrame): DataFrame containing pixel indices and other relevant data.

    Returns:
        gpd.GeoDataFrame: GeoDataFrame containing polygons representing clusters.
    """
    # Create an array to store cluster labels mapped to raster pixels
    cluster_raster = np.full(shape, -1, dtype=np.int32)
    cluster_raster[df["i"], df["j"]] = labels

    polygons = []
    for cluster_label in np.unique(labels):
        if cluster_label == -1:
            continue  # Skip noise points

        # Identify the mask for the current cluster
        cluster_mask = cluster_raster == cluster_label
        contours = measure.find_contours(cluster_mask, level=0.5)

        # Convert contours to polygons
        for contour in contours:
            polygon_coords = [
                transform * (col, row) for row, col in contour
            ]
            polygon = Polygon(polygon_coords)
            if polygon.is_valid:
                polygons.append((cluster_label, polygon))

    # Create a GeoDataFrame from the polygons
    gdf = gpd.GeoDataFrame(polygons, columns=["cluster_label", "geometry"], crs=crs)
    return gdf


def create_min_enclosing_polygons(gdf):
    """
    Generates minimum enclosing polygons for each cluster in the GeoDataFrame.

    Args:
        gdf (gpd.GeoDataFrame): GeoDataFrame containing cluster polygons.

    Returns:
        gpd.GeoDataFrame: GeoDataFrame with updated minimum enclosing polygons.
    """
    enclosing_polygons = []

    for cluster_label in gdf["cluster_label"].unique():
        # Combine all polygons for the same cluster label into a single geometry
        cluster_polygons = gdf[gdf["cluster_label"] == cluster_label]
        combined_geometry = unary_union(cluster_polygons.geometry)

        # Create the minimum enclosing polygon (remove self-intersections with buffer)
        enclosing_polygon = combined_geometry.buffer(0)
        enclosing_polygons.append((cluster_label, enclosing_polygon))

    # Create a new GeoDataFrame with the enclosing polygons
    gdf_enclosing = gpd.GeoDataFrame(enclosing_polygons, columns=["cluster_label", "geometry"], crs=gdf.crs)
    return gdf_enclosing


# Calling the function with appropriate parameters
dbscan_flood_areas = process_raster_to_clusters(
    threshold_m=flood_depth_threshold,  # Minimum depth threshold
    eps=eps_threshold,            # DBSCAN epsilon value
    raster_file=raster_file,    # Input raster file path
    minPts=5
)

# Saving the output to a GeoPackage file
layer_name = f"STEP_1_dbscan_cluster"
add_Layer_to_File(dbscan_flood_areas, processing_file, layer_name, "GPKG")


In [104]:
def generate_polygons_along_edge(edge_layer, node_layer, buffer, step_size=0.1):
    bounding_boxes = []
    distances = []
    angles = []
    
    for _, edge in edge_layer.iterrows():
        from_node_id = edge['from_id']
        to_node_id = edge['to_id']
        from_node = node_layer[node_layer['node_id'] == from_node_id].geometry.iloc[0]
        to_node = node_layer[node_layer['node_id'] == to_node_id].geometry.iloc[0]
        
        if (round(from_node.x, 5) == round(to_node.x, 5)) and (round(from_node.y, 5) == round(to_node.y, 5)):
            continue
            
        distance = from_node.distance(to_node)
        midpoint = calculate_midpoint(from_node, to_node)
        angle = calculate_angle(from_node, to_node)
        
        current_buffer = 0.5  # Ensure the first step is always added
        best_bounding_box = None
        final_bounding_box = None
        
        while current_buffer <= inland_buffer_distance:
            min_y = min(from_node.y, to_node.y)
            max_y = max(from_node.y, to_node.y)
            height = max_y - min_y + (current_buffer * 1000)
            min_x = min(from_node.x, to_node.x)
            max_x = min_x + distance
            
            bounding_box = Polygon([
                (min_x, min_y),
                (min_x, min_y + height),
                (max_x, min_y + height),
                (max_x, min_y),
                (min_x, min_y)
            ])
            
            translated_bounding_box = translate_bounding_box(bounding_box, midpoint)
            rotated_bounding_box = rotate_bounding_box(translated_bounding_box, midpoint, angle)
            
            # Check number of intersection points with Jamaica's boundary
            boundary_intersection = rotated_bounding_box.boundary.intersection(jamaica_polygon_revised.geometry.union_all().boundary)
            num_intersection_points = 0
            
            # Count the number of intersection points
            if boundary_intersection.geom_type == 'Point':
                num_intersection_points = 1
            elif boundary_intersection.geom_type == 'MultiPoint':
                num_intersection_points = len(boundary_intersection.geoms)
            elif boundary_intersection.geom_type == 'GeometryCollection':
                # Count points in a GeometryCollection
                num_intersection_points = sum(1 for geom in boundary_intersection.geoms if geom.geom_type == 'Point')
            elif boundary_intersection.geom_type == 'LineString':
                # LineString means multiple intersection points along a path, count as more than 2
                num_intersection_points = 3  # Just to trigger the break condition
            elif boundary_intersection.geom_type == 'MultiLineString':
                # Multiple linestrings also indicate more than 2 intersection points
                num_intersection_points = 3  # Just to trigger the break condition
            
            # New stopping condition: only stop if intersection points > 2
            if num_intersection_points <= 2:
                best_bounding_box = rotated_bounding_box
            else:
                break  # Stop increasing the buffer if more than 2 intersection points
                
            current_buffer += step_size  # Increase the buffer
            
        final_bounding_box = best_bounding_box if best_bounding_box is not None else rotated_bounding_box
        
        if final_bounding_box is not None:
            bounding_boxes.append(final_bounding_box)
            distances.append(distance)
            angles.append(angle)
    
    # Create the bounding boxes GeoDataFrame
    bounding_boxes_gdf = gpd.GeoDataFrame(geometry=bounding_boxes, crs=edge_layer.crs)
    
    # Add ID column to the bounding boxes GeoDataFrame
    bounding_boxes_gdf['id'] = range(len(bounding_boxes_gdf))
    
    # Add rectangle_id to the edge layer to match the bounding boxes
    edge_layer_copy = edge_layer.copy()
    edge_layer_copy['rectangle_id'] = range(len(bounding_boxes_gdf))
    
    # Additional attributes
    bounding_boxes_gdf['distance'] = distances
    bounding_boxes_gdf['angle'] = angles
    
    return bounding_boxes_gdf, edge_layer_copy

def calculate_midpoint(from_node, to_node):
    return Point((from_node.x + to_node.x) / 2, (from_node.y + to_node.y) / 2)

def calculate_angle(from_node, to_node):
    delta_x = to_node.x - from_node.x
    delta_y = to_node.y - from_node.y
    return math.degrees(math.atan2(delta_y, delta_x))

def translate_bounding_box(bounding_box, midpoint):
    current_center_x = (bounding_box.bounds[0] + bounding_box.bounds[2]) / 2
    current_center_y = (bounding_box.bounds[1] + bounding_box.bounds[3]) / 2
    return affinity.translate(bounding_box, xoff=midpoint.x - current_center_x, yoff=midpoint.y - current_center_y)

def rotate_bounding_box(bounding_box, midpoint, angle):
    return affinity.rotate(bounding_box, angle, origin=(midpoint.x, midpoint.y))

# Example usage
edge_polygons, coastline_edge_layer = generate_polygons_along_edge(
    edge_layer=coastline_edge_layer,
    node_layer=coastline_node_layer,
    buffer=inland_buffer_distance,
)

# Save the output to a file
layer_name = f"STEP_2a_edge_polygons"
add_Layer_to_File(edge_polygons, processing_file, layer_name, "GPKG")

# layer_name = "edges"
# add_Layer_to_File(coastline_edge_layer, input_file, layer_name, "GPKG")


In [106]:
def add_max_zonal_stats(geopkg, raster_path, new_column_name):
    """
    Add a new column to a GeoDataFrame with the maximum raster value for each polygon,
    mimicking the functionality of the Zonal Statistics Tool in QGIS. 
    The GeoDataFrame is reprojected to match the raster CRS during the calculation 
    and reprojected back to its original CRS afterward.

    Parameters:
    - geopkg (GeoDataFrame): GeoDataFrame containing polygons for which zonal statistics will be calculated.
    - raster_path (str): Path to the raster file used for calculating zonal statistics.
    - new_column_name (str): Name of the new column to store the maximum raster value for each polygon.

    Returns:
    - GeoDataFrame: The updated GeoDataFrame with the new column containing maximum raster values.
    """
    # Save the original CRS (Coordinate Reference System) of the GeoDataFrame for re-projection later
    original_crs = geopkg.crs

    # Open the raster file to retrieve its CRS
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    # Reproject the GeoDataFrame to match the raster CRS if their CRS do not align
    if geopkg.crs != raster_crs:
        geopkg = geopkg.to_crs(raster_crs)


    geopkg_temp = geopkg.copy()
    # Ensure all geometries are valid (use a buffer of 0 as a fix for invalid geometries)
    geopkg_temp["geometry"] = geopkg_temp["geometry"].buffer(50)

    # Calculate zonal statistics to determine the maximum raster value within each polygon
    stats = zonal_stats(
        geopkg_temp,          # GeoDataFrame containing polygons
        raster_path,     # Path to the raster file
        stats="max",     # Calculate the maximum value for each zone
        geojson_out=False,  # Do not output results in GeoJSON format
        nodata=-9999     # Value to treat as NoData in the raster (adjust as necessary)
    )

    # Extract the maximum raster values from the zonal statistics results and add to a new column
    geopkg[new_column_name] = [
        stat["max"] if stat["max"] is not None else None  # Add max value or None if no data exists
        for stat in stats
    ]

    # Reproject the GeoDataFrame back to its original CRS
    if geopkg.crs != original_crs:
        geopkg = geopkg.to_crs(original_crs)

    return geopkg  # Return the updated GeoDataFrame with the new column


# Call the function to calculate and add the maximum flood height from the raster
edge_polygons = add_max_zonal_stats(
    geopkg=edge_polygons,
    raster_path=raster_file,
    new_column_name="max_flood_height"
)

# Save the updated GeoDataFrame back to the file, ensuring CRS consistency
layer_name = f"STEP_2a_edge_polygons"
add_Layer_to_File(edge_polygons, processing_file, layer_name, "GPKG")


In [118]:
def join_polygons_by_flood_height(gdf, flood_height_threshold, length_limit, group_size):
    """
    Groups polygons in a GeoDataFrame based on their maximum flood height and coastline length.
    
    Parameters:
    - gdf (GeoDataFrame): Input GeoDataFrame containing polygons and their associated maximum flood heights
    - flood_height_threshold (float): The flood height threshold for grouping polygons
    - length_limit (float): The coastline length limit for groups (meters)
    - group_size (int): The minimum number of polygons needed to make up a group

    Returns:
    - GeoDataFrame: A new GeoDataFrame containing the grouped polygons
    """
    
    # Initialize lists to store results
    groups = []          # Combined geometries
    heights = []         # Maximum flood heights for each group
    ids = []             # Original IDs in each group
    
    # Initialize tracking variables
    current_group = []
    current_heights = []
    current_ids = []
    current_threshold_state = None  # None, True (above), or False (below)
    
    # Function to check coastline length for a geometry
    def get_coastline_length(geom):
        # Buffer the geometry slightly to ensure intersection with coastline
        buffered_geom = geom.buffer(10)
        # Find intersection with coastline
        coast_intersection = full_coastline_layer.geometry.iloc[0].intersection(buffered_geom)
        # Calculate length of the intersection
        if coast_intersection.is_empty:
            return 0
        elif hasattr(coast_intersection, 'geoms'):  # MultiLineString
            return sum(line.length for line in coast_intersection.geoms)
        else:  # Single LineString
            return coast_intersection.length
    
    # Function to finalize a group
    def finalize_group():
        nonlocal current_group, current_heights, current_ids
        
        if not current_group:
            return
            
        # Check if group is too small and there are previous groups
        if len(current_group) < group_size and groups:
            # Add small group to previous group
            groups[-1] = unary_union([groups[-1]] + current_group).convex_hull
            heights[-1] = max(heights[-1], max(current_heights))
            ids[-1].extend(current_ids)
        else:
            # Create new group
            combined_geom = unary_union(current_group).convex_hull
            groups.append(combined_geom)
            heights.append(max(current_heights))
            ids.append(current_ids.copy())
        
        # Reset tracking variables
        current_group = []
        current_heights = []
        current_ids = []
    
    # Process each polygon in the input GeoDataFrame
    for idx, row in gdf.iterrows():
        # Determine if polygon is above threshold
        is_above = row["max_flood_height"] > flood_height_threshold
        
        # Check if we need to start a new group based on threshold
        if (current_threshold_state is not None and 
            current_threshold_state != is_above):
            finalize_group()
            current_threshold_state = is_above
        
        # If first polygon or empty group, set threshold state
        if current_threshold_state is None:
            current_threshold_state = is_above
        
        # Check if adding this polygon would exceed length limit
        if current_group:
            # Calculate potential new geometry
            potential_geom = unary_union(current_group + [row["geometry"]])
            coastline_length = get_coastline_length(potential_geom)
            
            # Start new group if length limit would be exceeded
            if coastline_length > length_limit:
                finalize_group()
                current_threshold_state = is_above
        
        # Add polygon to current group
        current_group.append(row["geometry"])
        current_heights.append(row["max_flood_height"])
        current_ids.append(row["id"])
    
    # Finalize the last group
    finalize_group()
    
    # Create result GeoDataFrame
    result_gdf = gpd.GeoDataFrame(
        {
            "geometry": gpd.GeoSeries(groups),
            "max_flood_height": heights,
            "original_ids": ids,
        }
    )
    
    # Add unique IDs
    result_gdf["id"] = range(len(result_gdf))
    
    # Set CRS to match input
    result_gdf.set_crs(gdf.crs, inplace=True)
    
    return result_gdf

def order_edges_around_island(edge_layer):
    """
    Order edges around an island, starting with edge_0 and following the
    connections based on from_id and to_id.

    Parameters:
    - edge_layer (GeoDataFrame): GeoDataFrame containing edges with 'id', 'from_id', and 'to_id' columns

    Returns:
    - List[str]: List of associated bounding box IDs in the order the edges are traced around the island
    """
    # Initialize the ordered list with the first edge (edge_0)
    ordered_edges = []

    # Create a dictionary to lookup edges by their 'from_id' for efficient access
    edges_by_from_id = {
        edge['from_id']: edge
        for _, edge in edge_layer.iterrows()
    }

    # Start with the edge having 'id' equal to 'edge_0'
    current_edge = edge_layer[edge_layer['id'] == 'edge_0'].iloc[0]
    ordered_edges.append(current_edge['rectangle_id'])  # Append the first edge's ID

    # Track the starting node to detect when we've completed a loop
    start_node = current_edge['from_id']

    # Continue looping through the edges, following the 'to_id' until we return to the starting node
    while True:
        # Find the next edge based on the current edge's 'to_id'
        next_from_id = current_edge['to_id']

        # Break the loop if we return to the starting node (complete the loop)
        if next_from_id == start_node:
            break

        # Look up the next edge using the 'from_id' as the key in the dictionary
        current_edge = edges_by_from_id[next_from_id]
        ordered_edges.append(current_edge['rectangle_id'])  # Add the next edge's ID to the ordered list

    return ordered_edges  # Return the ordered list of edge IDs

# Call the function to get the ordered list of edge IDs
ordered_edge_ids = order_edges_around_island(
    edge_layer = coastline_edge_layer
)

# Filter the original polygon layer to include only the ordered edge IDs
ordered_edge_ids = [eid for eid in ordered_edge_ids if eid in edge_polygons["id"].values]
input_polygons = edge_polygons.set_index("id").loc[ordered_edge_ids].reset_index()


# Parameters for grouping polygons
flood_height_threshold = 0.1  # Flood height threshold above which polygons will be grouped
length_limit = 5_000  # Area limit for group finalization
group_size = 5  # Minimum number of polygons needed to make up a group

# Call the function to join polygons based on flood height and area conditions
joined_polygons = join_polygons_by_flood_height(
    gdf = input_polygons, 
    flood_height_threshold = 0.1, 
    length_limit = length_limit, 
    group_size = group_size
)

# Define layer name for the final output
layer_name = f"STEP_2b_joined_polygons"

# Save the result back to a file, ensuring CRS consistency
add_Layer_to_File(joined_polygons, processing_file, layer_name, "GPKG")

In [134]:
def merge_overlapping_polygons(gdf, overlap_threshold):
    """
    Iteratively merges overlapping polygons that overlap by more than the specified threshold.
    
    Args:
        gdf (GeoDataFrame): GeoDataFrame containing polygons with 'geometry' and 'max_flood_height' fields.
        overlap_threshold (float): The minimum percentage of overlap required to trigger a merge (e.g., 0.35 for 35% overlap).
    
    Returns:
        GeoDataFrame: A new GeoDataFrame with merged polygons until no pair overlaps by more than the threshold.
    """
    
    # Ensure that the CRS (Coordinate Reference System) is defined for the input data
    if gdf.crs is None:
        gdf.set_crs("EPSG:3097", inplace=True)  # Set CRS to Jamaica Metric Grid, adjust if necessary

    # Helper function to perform one pass of merging overlapping polygons
    def merge_once(gdf, overlap_threshold):
        """
        Performs one pass over the GeoDataFrame to merge overlapping polygons.
        Merges polygons whose intersection area exceeds the given overlap threshold.
        
        Args:
            gdf (GeoDataFrame): GeoDataFrame to process.
            overlap_threshold (float): The overlap threshold to trigger a merge.
        
        Returns:
            GeoDataFrame: A new GeoDataFrame with merged polygons from this pass.
        """
        merged = []  # List to hold merged geometries
        indices_to_merge = set()  # Set to keep track of merged polygons

        # Loop through all polygons in the GeoDataFrame to check for overlaps
        for i, geom1 in enumerate(gdf.geometry):
            if i in indices_to_merge:
                continue  # Skip geometries that are already merged

            for j, geom2 in enumerate(gdf.geometry):
                if i >= j or j in indices_to_merge:
                    continue  # Avoid redundant checks and already-merged geometries

                # Check if the two polygons intersect
                if geom1.intersects(geom2):
                    intersection = geom1.intersection(geom2)
                    if intersection.is_empty:
                        continue  # Skip if the intersection is empty

                    # Calculate the area of the intersection and compare with the polygons' areas
                    area1 = geom1.area
                    area2 = geom2.area
                    intersection_area = intersection.area

                    # Check if the overlap exceeds the threshold
                    if (intersection_area / min(area1, area2)) > overlap_threshold:
                        # Merge the two polygons by creating a convex hull around the union
                        new_geom = unary_union([geom1, geom2]).convex_hull
                        new_height = max(gdf.loc[i, "max_flood_height"], gdf.loc[j, "max_flood_height"])

                        # Store the merged geometry and its max flood height
                        merged.append({
                            "geometry": new_geom,
                            "max_flood_height": new_height
                        })
                        indices_to_merge.update([i, j])  # Mark the merged polygons

                        break  # Stop checking other polygons for the current geometry once merged

        # Retain the geometries that were not merged
        non_merged = gdf.loc[~gdf.index.isin(indices_to_merge)]

        # Combine the merged geometries with the non-overlapping ones
        merged_gdf = gpd.GeoDataFrame(
            {
                "geometry": [m["geometry"] for m in merged],
                "max_flood_height": [m["max_flood_height"] for m in merged],
            }
        )

        # Ensure CRS is explicitly set for the merged geometries
        merged_gdf.set_crs(gdf.crs, inplace=True)

        # Concatenate the non-overlapping geometries with the merged ones
        final_gdf = gpd.GeoDataFrame(pd.concat([non_merged, merged_gdf], ignore_index=True))

        # Set CRS again after concatenation to ensure consistency
        final_gdf.set_crs(gdf.crs, inplace=True)

        return final_gdf

    # Iteratively merge polygons until no more overlaps are found
    prev_gdf = gdf
    while True:
        new_gdf = merge_once(prev_gdf, overlap_threshold)
        if len(new_gdf) == len(prev_gdf):  # No changes (no more overlaps)
            break
        prev_gdf = new_gdf  # Update the GeoDataFrame for the next iteration

    # Ensure final CRS consistency
    prev_gdf.set_crs(gdf.crs, inplace=True)

    # Reassign unique IDs starting from 1 for the merged polygons
    prev_gdf = prev_gdf.reset_index(drop=True)  # Reset the index
    prev_gdf['id'] = prev_gdf.index + 1  # Assign new unique IDs based on the index

    prev_gdf["length"] = prev_gdf.geometry.apply(lambda geom: longest_bounding_box_side(geom, prev_gdf.crs))

    return prev_gdf

def get_intersections_by_id(gdf, id_column="id"):
    """
    Creates a dictionary where the key is the polygon ID, and the value is a list of IDs
    of polygons that intersect with it.

    Args:
        gdf (GeoDataFrame): GeoDataFrame containing polygons with 'geometry'.
        id_column (str): The column name containing unique polygon IDs.

    Returns:
        dict: A dictionary where keys are polygon IDs and values are lists of intersecting polygon IDs.
    """
    intersections = {}

    # Ensure the CRS is defined
    if gdf.crs is None:
        raise ValueError("GeoDataFrame must have a defined CRS.")

    # Iterate over all pairs of polygons
    for i, geom1 in gdf.iterrows():
        polygon_id = geom1[id_column]
        intersecting_ids = []

        for j, geom2 in gdf.iterrows():
            other_id = geom2[id_column]
            
            # Skip self-comparison
            if polygon_id == other_id:
                continue
            
            # Check if the polygons intersect
            if geom1.geometry.intersects(geom2.geometry):
                intersecting_ids.append(other_id)

        # Update the dictionary
        intersections[polygon_id] = intersecting_ids

    return intersections


def clip_larger_polygons(gdf, id_column="id"):
    """
    Clips larger polygons to remove intersections with smaller polygons based on the intersection dictionary.

    Args:
        gdf (GeoDataFrame): GeoDataFrame containing polygons with 'geometry'.
        id_column (str): The column name containing unique polygon IDs.

    Returns:
        GeoDataFrame: Updated GeoDataFrame with clipped polygons.
    """
    # Ensure the CRS is defined
    if gdf.crs is None:
        raise ValueError("GeoDataFrame must have a defined CRS.")

    # Create the intersection dictionary
    intersections = get_intersections_by_id(gdf, id_column=id_column)
    
    # Iterate through the dictionary
    for polygon_id, intersecting_ids in intersections.items():
        # Get the geometry of the current polygon
        geom1 = gdf.loc[gdf[id_column] == polygon_id, 'geometry'].values[0]

        # Check each intersecting polygon
        for other_id in intersecting_ids:
            # Get the geometry of the other polygon
            geom2 = gdf.loc[gdf[id_column] == other_id, 'geometry'].values[0]

            # Check if they still intersect
            if geom1.intersects(geom2):
                # Calculate the areas of the polygons
                area1 = geom1.area
                area2 = geom2.area

                # Clip the larger polygon by subtracting the intersection
                if area1 >= area2:
                    clipped_geom = geom1.difference(geom2)
                    geom1 = extract_polygons(clipped_geom)  # Ensure only polygons
                else:
                    clipped_geom = geom2.difference(geom1)
                    geom2 = extract_polygons(clipped_geom)  # Ensure only polygons

                # Update the geometries in the GeoDataFrame
                gdf.loc[gdf[id_column] == polygon_id, 'geometry'] = geom1
                gdf.loc[gdf[id_column] == other_id, 'geometry'] = geom2

    # Reset index for consistency
    gdf = gdf.reset_index(drop=True)
    return gdf

def extract_polygons(geometry):
    """
    Extracts only polygonal geometries from a GeometryCollection or MultiPolygon.

    Args:
        geometry (shapely.geometry): Input geometry.

    Returns:
        shapely.geometry.Polygon or MultiPolygon: Cleaned geometry.
    """
    if isinstance(geometry, Polygon) or isinstance(geometry, MultiPolygon):
        return geometry
    elif geometry.is_empty:
        return None  # Return None for empty geometries
    elif hasattr(geometry, 'geoms'):  # Handle GeometryCollection
        polygons = [geom for geom in geometry.geoms if isinstance(geom, (Polygon, MultiPolygon))]
        if len(polygons) == 1:
            return polygons[0]  # Return as a single Polygon
        elif len(polygons) > 1:
            return MultiPolygon(polygons)  # Return as MultiPolygon
    return None  # In case no valid polygon is found

def longest_bounding_box_side(polygon, crs):
    """
    Computes the longest side of the minimum bounding box in meters.
    Ensures the bounding box covers the entire MultiPolygon.

    Parameters:
    - polygon: shapely.geometry.Polygon or MultiPolygon
    - crs: Coordinate Reference System (from the source GeoDataFrame)

    Returns:
    - The longest side of the bounding box in meters.
    """

    # Ensure the CRS is projected
    if crs is None or not crs.is_projected:
        raise ValueError("The provided CRS must be projected with units in meters.")

    # If it's a MultiPolygon, merge into a single shape
    if isinstance(polygon, MultiPolygon):
        polygon = polygon.convex_hull  # Create a convex hull around all parts

    # Get the minimum rotated rectangle
    min_rect = polygon.minimum_rotated_rectangle
    coords = list(min_rect.exterior.coords)  # Extract rectangle coordinates

    # Compute side lengths
    side_lengths = [((coords[i][0] - coords[i+1][0])**2 + (coords[i][1] - coords[i+1][1])**2) ** 0.5 
                    for i in range(4)]  # Only 4 sides in a rectangle

    return max(side_lengths)

# Main execution code
overlap = 0.50

# Apply the merging function to resolve overlaps
s_cape_flood_areas = merge_overlapping_polygons(
    gdf = joined_polygons, 
    overlap_threshold = overlap
)

s_cape_flood_areas = clip_larger_polygons(s_cape_flood_areas, "id")

# Create a layer name dynamically for the merged flood areas
layer_name = f"STEP_2c_scape_cluster"

# Add the merged polygons layer to the file
add_Layer_to_File(s_cape_flood_areas, processing_file, layer_name, "GPKG")

In [135]:
# def merge_overlapping_polygons(gdf, length_limit):
#     """
#     Iteratively merges overlapping polygons if the merged coastline length is within the specified limit.
    
#     Args:
#         gdf (GeoDataFrame): GeoDataFrame containing polygons with 'geometry' and 'max_flood_height' fields.
#         length_limit (float): The maximum allowed coastline length for merged polygons.
    
#     Returns:
#         GeoDataFrame: A new GeoDataFrame with merged polygons until no pair exceeds the length limit.
#     """
    
#     if gdf.crs is None:
#         gdf.set_crs("EPSG:3097", inplace=True)

#     def merge_once(gdf, length_limit):
#         """
#         Performs one pass over the GeoDataFrame to merge overlapping polygons.
#         Merges polygons if the resulting coastline length does not exceed the limit.
        
#         Args:
#             gdf (GeoDataFrame): GeoDataFrame to process.
#             length_limit (float): The maximum allowed coastline length for merged polygons.
        
#         Returns:
#             GeoDataFrame: A new GeoDataFrame with merged polygons from this pass.
#         """
#         merged = []
#         indices_to_merge = set()

#         for i, geom1 in enumerate(gdf.geometry):
#             if i in indices_to_merge:
#                 continue

#             for j, geom2 in enumerate(gdf.geometry):
#                 if i >= j or j in indices_to_merge:
#                     continue

#                 if geom1.intersects(geom2):
#                     new_geom = unary_union([geom1, geom2]).convex_hull
#                     new_length = get_coastline_length(new_geom)

#                     if new_length <= length_limit:
#                         new_height = max(gdf.loc[i, "max_flood_height"], gdf.loc[j, "max_flood_height"])
#                         merged.append({
#                             "geometry": new_geom,
#                             "max_flood_height": new_height
#                         })
#                         indices_to_merge.update([i, j])
#                         break

#         non_merged = gdf.loc[~gdf.index.isin(indices_to_merge)]

#         merged_gdf = gpd.GeoDataFrame(
#             {
#                 "geometry": [m["geometry"] for m in merged],
#                 "max_flood_height": [m["max_flood_height"] for m in merged],
#             }
#         )

#         merged_gdf.set_crs(gdf.crs, inplace=True)
#         final_gdf = gpd.GeoDataFrame(pd.concat([non_merged, merged_gdf], ignore_index=True))
#         final_gdf.set_crs(gdf.crs, inplace=True)

#         return final_gdf

#     prev_gdf = gdf
#     while True:
#         new_gdf = merge_once(prev_gdf, length_limit)
#         if len(new_gdf) == len(prev_gdf):
#             break
#         prev_gdf = new_gdf

#     prev_gdf.set_crs(gdf.crs, inplace=True)
#     prev_gdf = prev_gdf.reset_index(drop=True)
#     prev_gdf['id'] = prev_gdf.index + 1
#     prev_gdf["length"] = prev_gdf.geometry.apply(lambda geom: longest_bounding_box_side(geom, prev_gdf.crs))

#     return prev_gdf

# def get_coastline_length(geom):
#     """
#     Calculates the length of the coastline segment intersecting with a given geometry.
    
#     Args:
#         geom (shapely.geometry): The input polygon geometry.
    
#     Returns:
#         float: The length of the coastline segment.
#     """
#     buffered_geom = geom.buffer(10)
#     coast_intersection = full_coastline_layer.geometry.iloc[0].intersection(buffered_geom)
    
#     if coast_intersection.is_empty:
#         return 0
#     elif hasattr(coast_intersection, 'geoms'):
#         return sum(line.length for line in coast_intersection.geoms)
#     else:
#         return coast_intersection.length

# # Main execution code
# length_limit = 5_000  # Example length limit in meters

# s_cape_flood_areas = merge_overlapping_polygons(
#     gdf = s_cape_flood_areas, 
#     length_limit = length_limit
# )

# s_cape_flood_areas = clip_larger_polygons(s_cape_flood_areas, "id")

# layer_name = f"STEP_2c_scape_cluster_coastline"
# add_Layer_to_File(s_cape_flood_areas, processing_file, layer_name, "GPKG")


In [136]:
def create_jamaica_convex_hull(buffer_distance, edge_layer):
    def create_trimmed_convex_hull(polygons):
        """
        Create a convex hull encompassing the given polygons by computing the convex hull
        from their union.
        
        Args:
            polygons (list): A list of geometries (polygons) to create the convex hull around.
        
        Returns:
            geometry: The convex hull geometry encompassing all the input polygons.
        """
        # Combine the polygons and compute the convex hull of their union
        combined = unary_union(polygons)  # Union of all polygons
        convex_hull = combined.convex_hull  # Convex hull of the union

        return convex_hull

    # Create a GeoDataFrame containing the union of all coastline geometries
    coast_linestring = gpd.GeoDataFrame({"id": [0], "geometry": [edge_layer.geometry.union_all()]})
    coast_linestring.crs = edge_layer.crs
    buffered_coastline = coast_linestring.buffer(buffer_distance * 1_000)
    # return buffered_coastline

    # Ensure the geometry column exists and convert to a list of geometries
    polygons = buffered_coastline.geometry.tolist()  # List of geometries for hull creation

    # Create the trimmed convex hull using the helper function
    trimmed_hull = create_trimmed_convex_hull(polygons)

    # Prepare a new GeoDataFrame for the trimmed convex hull
    jam_convex = gpd.GeoDataFrame({
        'id': ['jam_convex'],  # Assign an ID to the new hull
        'geometry': [trimmed_hull]  # The geometry of the convex hull
    }, crs=buffered_coastline.crs)  # Ensure the new row uses the same CRS as the input GeoDataFrame

    return jam_convex

jamaica_convex_hull = gpd.GeoDataFrame({
        'id': [1],  # Assign an ID to the new hull
        'geometry': jamaica_polygon_revised.buffer(10000)  # The geometry of the convex hull
    }, crs=coastline_edge_layer.crs)

# Define the layer name for the convex hull and save it to the GeoPackage
layer_name = "STEP_3_processing_area"
add_Layer_to_File(jamaica_convex_hull, processing_file, layer_name, "GPKG")


In [137]:
dbscan_flood_areas_all = process_raster_to_clusters(
    threshold_m=flood_depth_threshold,  # Minimum depth threshold
    eps=eps_threshold,            # DBSCAN epsilon value
    raster_file=raster_file,    # Input raster file path
    minPts=5
)

In [138]:
# Adapted and modified from https://github.com/thomas-fred/jam-coastal-protection

def combine_floods_within_scape(voronoi_polygons: gpd.GeoDataFrame,flood_polygons: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    combined_polygons = []  # List to store resulting geometries
    voronoi_ids = []  # List to store Voronoi IDs (optional, for tracking)

    # Loop through each Voronoi polygon
    for voronoi_index, voronoi in voronoi_polygons.iterrows():
        voronoi_geom = voronoi.geometry

        # Find all flood polygons that intersect with the current Voronoi polygon
        intersecting_floods = flood_polygons[flood_polygons.intersects(voronoi_geom)]

        if not intersecting_floods.empty:
            # Combine the intersecting flood polygons within the Voronoi polygon
            combined_geom = intersecting_floods.intersection(voronoi_geom).union_all()

            if combined_geom:
                # Append the filtered geometry and its Voronoi index
                combined_polygons.append(combined_geom)
                voronoi_ids.append(voronoi['id'])
        else:
            combined_polygons.append(None)
            voronoi_ids.append(voronoi['id'])

    # Create a GeoDataFrame for the combined polygons
    result_gdf = gpd.GeoDataFrame({
        "id": voronoi_ids,  # Include the Voronoi IDs
        "geometry": combined_polygons  # Include the geometries
    }, crs=voronoi_polygons.crs)  # Set CRS to match Voronoi polygons

    return result_gdf  # Return the combined flood polygons GeoDataFrame

# Combine flood polygons for each Voronoi polygon using the function above
scape_intersection = combine_floods_within_scape(
    voronoi_polygons = s_cape_flood_areas,  # Voronoi polygons GeoDataFrame
    flood_polygons = dbscan_flood_areas_all  # Flood polygons GeoDataFrame
)

# Add the resulting dbscan_voronoi_intersection layer to a processing file (GeoPackage format)
layer_name = "STEP_4_scape_flood_intersection"
add_Layer_to_File(scape_intersection, processing_file, layer_name, "GPKG")




In [153]:
def create_voronoi(jamaica_polygon, jam_contour, coastline_polygons, intersections):
    """
    Create Voronoi tessellation based on the centroids of smaller polygons (coastline),
    clipped to the boundary of a larger polygon (Jamaica's convex hull).
    
    Args:
        jamaica_polygon (GeoDataFrame): The larger polygon (e.g., Jamaica's convex hull).
        jam_contour (GeoDataFrame): A GeoDataFrame containing the exact contour of the Jamaica.
        coastline_polygons (GeoDataFrame): The smaller polygons (e.g., coastline polygons) for which Voronoi regions will be calculated.

    Returns:
        GeoDataFrame: Clipped Voronoi polygons inside the larger polygon with associated centroid IDs.
        GeoDataFrame: Centroids of the smaller polygons used for Voronoi tessellation with a 'c_id' column.
    """
    # Load the larger polygon (assumed to be Jamaica's convex hull)
    convex_gdf = jamaica_polygon
    larger_polygon = convex_gdf.iloc[0].geometry  # Assuming the first row contains the larger polygon

    # Load the coastline polygons (smaller polygons)
    gdf = coastline_polygons
    smaller_polygons = gdf

    # Ensure the CRS is consistent for all GeoDataFrames
    smaller_polygons.crs = gdf.crs  # Ensure the coordinate reference system is consistent

    #-------------------------------------------------
    # Extract centroids of the smaller polygons to serve as Voronoi seed points
    # def get_centroid(row, smaller_polygons):
    #     if row.geometry and not row.geometry.is_empty:
    #         return row.geometry.centroid
    #     # Find the corresponding geometry in smaller_polygons
    #     match = smaller_polygons[smaller_polygons["id"] == row["id"]]
    #     return match.geometry.iloc[0].centroid if not match.empty else None

    # # Compute centroids while handling missing geometries
    # centroids = intersections.apply(lambda row: get_centroid(row, smaller_polygons), axis=1)
    centroids = smaller_polygons.geometry.centroid
    #-----------------------------------------------------

    # Add a 'c_id' column to the centroids, which identifies the original polygon
    # centroid_ids = smaller_polygons['id'] if 'id' in smaller_polygons.columns else range(len(centroids))
        
    if 'id' in smaller_polygons.columns and not smaller_polygons['id'].isnull().any():
        centroid_ids = smaller_polygons['id'].values
    else:
        centroid_ids = range(len(centroids))

    centroid_gdf = gpd.GeoDataFrame({'c_id': centroid_ids,'s_id': centroid_ids }, geometry=centroids, crs=smaller_polygons.crs)


    # Function to adjust points to be within or touching a larger polygon
    def adjust_point_to_polygon(point, polygon):
        """
        Adjust the point to be within or touching the boundary of the larger polygon.
        If the point is outside, it will be moved to the nearest boundary point.
        
        Args:
            point (Point): The point (centroid) to adjust.
            polygon (Polygon): The larger polygon representing Jamaica's exact shape.
        
        Returns:
            Point: The adjusted point that is within or on the boundary of the polygon.
        """
        if polygon.contains(point) or polygon.touches(point):
            return point
        else:
            # Find the nearest point on the boundary
            nearest_boundary_point = nearest_points(point, polygon.boundary)[1]
            # Move the point slightly closer to the boundary
            adjusted_point = Point(
                point.x + (nearest_boundary_point.x - point.x),
                point.y + (nearest_boundary_point.y - point.y)
            )
            # Ensure the adjusted point is within or touching the polygon
            if polygon.contains(adjusted_point) or polygon.touches(adjusted_point):
                return adjusted_point
            # As a fallback, snap directly to the nearest boundary point
            return nearest_boundary_point

    # Adjust centroids to be within or touching the larger polygon (Jamaica's boundary)
    adjusted_centroids = [adjust_point_to_polygon(pt, jam_contour.iloc[0].geometry) for pt in centroids]

    # Convert the adjusted centroids back to a GeoSeries for consistent geometry
    adjusted_centroids_gs = gpd.GeoSeries(adjusted_centroids, crs=centroid_gdf.crs)

    # Update the GeoDataFrame with adjusted centroids
    centroid_gdf.geometry = adjusted_centroids_gs

    # return None, centroid_gdf

    # Create a MultiPoint object from the adjusted centroids for Voronoi tessellation
    seed_points = MultiPoint(adjusted_centroids)

    # Perform Voronoi tessellation using the adjusted centroids as seed points
    voronoi = voronoi_diagram(seed_points)

    #-------------------------------------------------
    # Clip the Voronoi regions to the boundary of the larger polygon (Jamaica's convex hull)
    clipped_regions = [region.intersection(larger_polygon) for region in voronoi.geoms]

    # Filter out invalid or empty regions, keeping only valid polygons
    valid_regions = [region for region in clipped_regions if not region.is_empty and isinstance(region, Polygon)]

    # Match centroids to Voronoi regions based on proximity (the closest centroid to each Voronoi region)
    associated_centroid_ids = []
    for region in valid_regions:
        # Find the centroid nearest to the region's centroid
        region_centroid = region.centroid
        nearest_centroid = centroids.distance(region_centroid).idxmin()
        associated_centroid_ids.append(centroid_gdf.loc[nearest_centroid, 'c_id'])

    # Create a GeoDataFrame for the clipped Voronoi regions with associated centroid IDs
    clipped_gdf = gpd.GeoDataFrame(
        {'id': range(len(valid_regions)),  # Assign a unique ID to each Voronoi region
         'centroid_id': associated_centroid_ids},  # Store the associated centroid ID for each region
        geometry=valid_regions,
        crs=smaller_polygons.crs  # Use the same CRS as the smaller polygons
    )

    clipped_gdf["coastal_length"] = clipped_gdf.geometry.apply(lambda poly: find_coastline_length(full_coastline_layer, poly))
    longest_polygon = clipped_gdf.loc[clipped_gdf["coastal_length"].idxmax()]

    # Return both the clipped Voronoi regions and the centroids GeoDataFrame
    return clipped_gdf, centroid_gdf

def find_coastline_length(coastline, polygon):
    # if coastline.geometry.iloc[0].intersects(polygon.geometry):
    intersection = coastline.geometry.iloc[0].intersection(polygon)
    return intersection.length

# Load contour and coastline polygons from the input files
jam_contour = gpd.read_file(input_file, layer="jam") 
# jamaica_convex_hull = gpd.read_file(input_file, layer="jamaica_convex")

# Create the Voronoi polygons clipped to Jamaica's convex hull
voronoi_polygons, voronoi_centroid_points = create_voronoi(
    jamaica_polygon = jamaica_convex_hull, 
    jam_contour = jam_contour,
    coastline_polygons = s_cape_flood_areas,
    intersections= scape_intersection
)

# Save the Voronoi polygons and centroid points to a GeoPackage
layer_name = "STEP_5a_voronoi_polygons"
add_Layer_to_File(voronoi_polygons, processing_file, layer_name, "GPKG")

layer_name = "STEP_5a_voronoi_seed_pts"
add_Layer_to_File(voronoi_centroid_points, processing_file, layer_name, "GPKG")


In [150]:
from shapely.geometry import LineString, MultiLineString
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString

# def split_geometry(geometry, num_splits):
#     """Splits a LineString or MultiLineString into approximately equal-length segments."""
#     total_length = geometry.length
#     segment_length = total_length / num_splits

#     def get_segment(line, start, end):
#         # Using interpolate to create a segment between two distances
#         return LineString([line.interpolate(start), line.interpolate(end)])

#     if isinstance(geometry, LineString):
#         segments = [get_segment(geometry, i * segment_length, (i + 1) * segment_length) for i in range(num_splits)]
#         return segments

#     elif isinstance(geometry, MultiLineString):
#         accumulated_length = 0
#         segments = []
        
#         for line in geometry.geoms:
#             line_length = line.length
#             while accumulated_length + line_length >= segment_length and len(segments) < num_splits:
#                 split_point = segment_length - accumulated_length
#                 first_part = get_segment(line, 0, split_point)
#                 second_part = get_segment(line, split_point, line_length)

#                 segments.append(first_part)
#                 line = second_part
#                 line_length = line.length
#                 accumulated_length = 0

#             accumulated_length += line_length

#         return MultiLineString(segments) if len(segments) > 1 else segments


#     raise ValueError("Input must be a LineString or MultiLineString")

def find_midpoints(geometry, num_splits):
    """Finds the midpoints of `num_splits` segments along a LineString or MultiLineString."""
    if not isinstance(geometry, (LineString, MultiLineString)):
        raise ValueError("Input must be a LineString or MultiLineString")
    
    total_length = geometry.length
    if total_length == 0:
        return []

    segment_length = total_length / num_splits
    return [geometry.interpolate((i + 0.5) * segment_length, normalized=False) for i in range(num_splits)]

def generate_extra_points(polygon, coastline, num_pts, scale_factor, jam_contour):
    """Generates points at segment midpoints from a split coastline within a polygon."""
    clipped = polygon.intersection(jam_contour.geometry.iloc[0])
    coast_intersection = coastline.geometry.iloc[0].intersection(polygon.buffer(10))
    # print(coast_intersection)

    if num_pts < 2:
        num_splits = 2
    else:
        num_splits = num_pts
    
    midpoints = find_midpoints(coast_intersection, num_splits)
    
    layers = {
        "STEP_5b_debug_test_line": [coast_intersection],
        "STEP_5b_debug_test_points": midpoints,
        "STEP_5b_poly": clipped
    }
    
    for name, geometries in layers.items():
        try:
            gdf = gpd.GeoDataFrame(geometry=geometries, crs=jamaica_polygon_revised.crs)
            add_Layer_to_File(gdf, processing_file, name, "GPKG")
        except:
            pass
    
    # print(midpoints)
    return midpoints

jam_contour = gpd.read_file(input_file, layer="jam") 

# Example usage
polygon_id = 68  # Change as needed
polygon_row = s_cape_flood_areas[s_cape_flood_areas['id'] == polygon_id].iloc[0]
polygon = polygon_row.geometry
# print (polygon.area)
length_limit = 2  # Number of segments to split into

areas = s_cape_flood_areas.geometry.area.values 
scale_factor = (np.mean(areas) / len(areas)) / 5
# print (k)


new_points = generate_extra_points(polygon, full_coastline_layer,length_limit, scale_factor, jam_contour)


In [154]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPoint
from shapely.ops import nearest_points
from scipy.spatial import Voronoi
import pandas as pd

def get_larger_polygon(jamaica_polygon):
    """Returns the larger polygon (Jamaica's convex hull)."""
    convex_gdf = jamaica_polygon
    return convex_gdf.iloc[0].geometry


def compute_centroids(intersections, smaller_polygons):
    """Compute centroids for the smaller polygons (coastline polygons)."""
    def get_centroid(row, smaller_polygons):
        if row.geometry and not row.geometry.is_empty:
            return row.geometry.centroid
        match = smaller_polygons[smaller_polygons["id"] == row["id"]]
        return match.geometry.iloc[0].centroid if not match.empty else None
    
    # return intersections.apply(lambda row: get_centroid(row, smaller_polygons), axis=1)
    return smaller_polygons.geometry.centroid


def create_centroid_gdf(centroids, smaller_polygons):
    """Create a GeoDataFrame with centroids and associated centroid IDs."""
    if 'id' in smaller_polygons.columns and not smaller_polygons['id'].isnull().any():
        centroid_ids = smaller_polygons['id'].values
    else:
        centroid_ids = range(len(centroids))

    return gpd.GeoDataFrame({'c_id': centroid_ids, 's_id': centroid_ids }, geometry=centroids, crs=smaller_polygons.crs)


def adjust_centroids_to_polygon(centroids, larger_polygon):
    """Adjust centroids to be within or touching the larger polygon's boundary."""
    adjusted_centroids = [adjust_point_to_polygon(pt, larger_polygon) for pt in centroids]
    return adjusted_centroids


def adjust_point_to_polygon(point, polygon):
    """Adjust a point to be within or touching the boundary of a polygon."""
    if polygon.contains(point) or polygon.touches(point):
        return point
    else:
        nearest_boundary_point = nearest_points(point, polygon.boundary)[1]
        adjusted_point = Point(
            point.x + (nearest_boundary_point.x - point.x),
            point.y + (nearest_boundary_point.y - point.y)
        )
        if polygon.contains(adjusted_point) or polygon.touches(adjusted_point):
            return adjusted_point
        return nearest_boundary_point


def create_voronoi_tessellation(centroids):
    """Create Voronoi tessellation based on centroids."""
    seed_points = MultiPoint(centroids)
    return voronoi_diagram(seed_points)


def clip_voronoi_to_polygon(voronoi, larger_polygon):
    """Clip Voronoi regions to the boundary of a larger polygon."""
    return [region.intersection(larger_polygon) for region in voronoi.geoms]


def associate_centroids_to_regions(valid_regions, centroids, centroid_gdf):
    """Associate centroids to the valid Voronoi regions based on proximity."""
    associated_centroid_ids = []
    associated_polygon_ids = []
    for region in valid_regions:
        region_centroid = region.centroid
        nearest_centroid = centroids.distance(region_centroid).idxmin()
        associated_centroid_ids.append(centroid_gdf.loc[nearest_centroid, 'c_id'])
        associated_polygon_ids.append(centroid_gdf.loc[nearest_centroid, 's_id'])
    return associated_centroid_ids, associated_polygon_ids


def create_clipped_voronoi_gdf(valid_regions, associated_centroid_ids, associated_polygon_ids, crs):
    """Create a GeoDataFrame for the clipped Voronoi regions with associated centroid IDs."""
    return gpd.GeoDataFrame(
        {'id': range(len(valid_regions)),
         'flood_zone_id': associated_polygon_ids,
         'centroid_id': associated_centroid_ids},
        geometry=valid_regions,
        crs=crs
    )


def find_coastline_length(coastline, polygon):
    """Find the length of the coastline segment that intersects with the polygon."""
    intersection = coastline.geometry.iloc[0].intersection(polygon)
    return intersection.length

def compute_bbox_length(coasline_polygon, voronoi_polygon, crs):
    intersection = coasline_polygon.intersection(voronoi_polygon)
    
    if intersection.is_empty:
        return 0  # No intersection found
    
    bounding_box = intersection.convex_hull.minimum_rotated_rectangle

    # Extract the coordinates of the bounding box
    coords = list(bounding_box.exterior.coords)

    # Compute distances between consecutive points
    distances = [np.sqrt((coords[i][0] - coords[i+1][0])**2 + (coords[i][1] - coords[i+1][1])**2) 
                 for i in range(len(coords) - 1)]
    
    longest_side = max(distances)  # Longest side of the bounding box
    
    # Save to file for debugging
    # add_Layer_to_File(gpd.GeoDataFrame({'geometry': [bounding_box]}, crs=crs), processing_file, "bbox", "GPKG")

    return longest_side
    
def create_voronoi(jamaica_polygon, jam_contour, coastline_polygons, intersections, length_limit, coastline):
    """
    Create Voronoi tessellation based on the centroids of smaller polygons (coastline),
    clipped to the boundary of a larger polygon (Jamaica's convex hull).
    
    Args:
        jamaica_polygon (GeoDataFrame): The larger polygon (e.g., Jamaica's convex hull).
        jam_contour (GeoDataFrame): A GeoDataFrame containing the exact contour of the Jamaica.
        coastline_polygons (GeoDataFrame): The smaller polygons (e.g., coastline polygons) for which Voronoi regions will be calculated.

    Returns:
        GeoDataFrame: Clipped Voronoi polygons inside the larger polygon with associated centroid IDs.
        GeoDataFrame: Centroids of the smaller polygons used for Voronoi tessellation with a 'c_id' column.
    """
    #Combine all coastline polygons into one polygon
    combined_coastline_polygons = unary_union(coastline_polygons.geometry)

    # Get the larger polygon (Jamaica's convex hull)
    larger_polygon = get_larger_polygon(jamaica_polygon)

    # Get the smaller coastline polygons
    smaller_polygons = coastline_polygons

    # Ensure the CRS is consistent for all GeoDataFrames
    smaller_polygons.crs = coastline_polygons.crs  # Ensure the coordinate reference system is consistent

    # Compute centroids for Voronoi tessellation
    centroids = compute_centroids(intersections, smaller_polygons)

    # Add centroid IDs to the centroids GeoDataFrame
    centroid_gdf = create_centroid_gdf(centroids, smaller_polygons)

    def process_voronoi(centroids, centroid_gdf, larger_polygon, smaller_polygons):
        adjusted_centroids = adjust_centroids_to_polygon(centroids, jam_contour.iloc[0].geometry.buffer(50)) # Adjust centroids to be within the larger polygon
        adjusted_centroids_gs = gpd.GeoSeries(adjusted_centroids, crs=centroid_gdf.crs) # Convert the adjusted centroids back to a GeoSeries
        centroid_gdf.geometry = adjusted_centroids_gs # Update the GeoDataFrame with adjusted centroids
        voronoi = create_voronoi_tessellation(adjusted_centroids) # Create Voronoi tessellation based on centroids
        clipped_regions = clip_voronoi_to_polygon(voronoi, larger_polygon) # Clip Voronoi regions to the boundary of the larger polygon
        valid_regions = [region for region in clipped_regions if not region.is_empty and isinstance(region, Polygon)]  # Filter out invalid or empty regions, keeping only valid polygons
        associated_centroid_ids, associated_polygon_ids = associate_centroids_to_regions(valid_regions, centroids, centroid_gdf) # Match centroids to Voronoi regions based on proximity
        clipped_gdf = create_clipped_voronoi_gdf(valid_regions,associated_centroid_ids ,associated_polygon_ids, smaller_polygons.crs) # Create GeoDataFrame for clipped Voronoi regions with associated centroid IDs
        return clipped_gdf, centroid_gdf

    clipped_gdf, centroid_gdf = process_voronoi(centroids, centroid_gdf, larger_polygon, smaller_polygons)
    clipped_gdf["coastal_length"] = clipped_gdf.geometry.apply(lambda poly: find_coastline_length(full_coastline_layer, poly))
    longest_polygon = clipped_gdf.loc[clipped_gdf["coastal_length"].idxmax()]     # Get the longest coastline polygon

    areas = coastline_polygons.geometry.area.values
    scale_factor = np.mean(areas) / len(areas)

    tracked_flood = []
    override = False

    while longest_polygon['coastal_length'] > length_limit or override:
        override = False
        # Select the polygon
        polygon_id = longest_polygon['centroid_id']
        scape_id = longest_polygon['flood_zone_id']
        # print (scape_id)
        polygon_row = coastline_polygons[coastline_polygons['id'] == scape_id].iloc[0]
        polygon = polygon_row.geometry

        # v_poly = polygon.buffer(50).intersection(longest_polygon.geometry)

        if scape_id not in tracked_flood:
            num_pts = round(coastline.intersection(polygon).iloc[0].length / length_limit) 
            # print (scape_id, num_pts)
            new_points = generate_extra_points(polygon, coastline, num_pts, scale_factor, jam_contour)
            centroid_gdf = centroid_gdf[centroid_gdf['c_id'] != polygon_id].reset_index(drop=True)
            
        else:
            centroid_gdf = centroid_gdf[centroid_gdf['s_id'] != scape_id].reset_index(drop=True)
            num_pts = len(clipped_gdf[clipped_gdf["flood_zone_id"] == scape_id])
            print (f"Removed old points for {scape_id} redoing  num_pts")
            num_pts += 1
            new_points = generate_extra_points(polygon, coastline, num_pts, scale_factor, jam_contour)

        if new_points:
            existing_c_ids = set(centroid_gdf['c_id'])
            start_c_id = max(existing_c_ids) + 1 if existing_c_ids else 1
            new_c_ids = [start_c_id + i for i in range(len(new_points))]

            new_centroid_gdf = pd.concat([
                centroid_gdf,
                gpd.GeoDataFrame(
                    {'c_id': new_c_ids, 's_id': [scape_id] * len(new_points)},  # s_id repeats for each new point
                    geometry=new_points, 
                    crs=centroid_gdf.crs
                )
            ], ignore_index=True)

            centroid_gdf = new_centroid_gdf
            

            centroids = centroid_gdf.geometry
            clipped_gdf, centroid_gdf = process_voronoi(centroids, centroid_gdf, larger_polygon, smaller_polygons)
            print(f"Added {len(new_points)} new points with c_ids {new_c_ids} for polygon {scape_id}")
            
            if scape_id not in tracked_flood:
                tracked_flood.append(scape_id)
                
            clipped_gdf["coastal_length"] = clipped_gdf.geometry.apply(lambda poly: find_coastline_length(full_coastline_layer, poly))
            longest_polygon = clipped_gdf.loc[clipped_gdf["coastal_length"].idxmax()]
            
            # print(f"Longest Polygon: {longest_polygon["id"]}    -   Length: {longest_polygon["coastal_length"]}")

            layer_name = "STEP_5a_voronoi_polygons"
            add_Layer_to_File(clipped_gdf, processing_file, layer_name, "GPKG")

            layer_name = "STEP_5a_voronoi_seed_pts"
            add_Layer_to_File(centroid_gdf, processing_file, layer_name, "GPKG")
            
            long_poly_coast = longest_polygon.geometry.buffer(50).intersection(full_coastline_layer.geometry.iloc[0])
            flood_area = coastline_polygons[coastline_polygons['id'] == scape_id].iloc[0].geometry
            matching_clipped_gdf = clipped_gdf[clipped_gdf["flood_zone_id"] == scape_id]
            
            # Check if long_poly_coast is NOT contained within its corresponding coastline polygon
            if not flood_area.contains(long_poly_coast):

                def is_intersection_contained(row):
                    intersection = row.geometry.intersection(full_coastline_layer.geometry.iloc[0])
                    return flood_area.contains(intersection)

                # Apply the check to filter only relevant polygons
                fully_contained = matching_clipped_gdf[matching_clipped_gdf.apply(is_intersection_contained, axis=1)]
                fraction_fully_contained = len(fully_contained) / len(matching_clipped_gdf) if len(matching_clipped_gdf) > 0 else 0
                # print (fraction_fully_contained)

                # Check if all fully contained polygons have a "coastal_length" < length_limit
                if not fully_contained.empty and (fully_contained["coastal_length"] < length_limit).all() and len(fully_contained) >= len(matching_clipped_gdf) - 2:
                    # Find all coastline polygons that intersect long_poly_coast
                    intersecting_polygons = coastline_polygons[coastline_polygons.geometry.intersects(long_poly_coast)]

                    # Exclude the current coastline polygon (scape_id) since we already checked it
                    other_intersecting = intersecting_polygons[intersecting_polygons['id'] != scape_id]

                    # Get the ID(s) of the intersecting polygons
                    if not other_intersecting.empty:
                        other_scape_id = other_intersecting.iloc[0]['id']  # Get the first match
                        print (f"----- Correcting edge case scenario for {other_scape_id} -----")
                        longest_polygon = clipped_gdf[clipped_gdf["flood_zone_id"] == other_scape_id].iloc[0]
                        override = True

                # Save test layer
                # add_Layer_to_File(gpd.GeoDataFrame([longest_polygon], crs = clipped_gdf.crs), processing_file, "condition_test", "GPKG")

        else:
            print (f"No points {new_points}, was polygon worked on? - {scape_id in tracked_flood}")
            

        
    # Return both the clipped Voronoi regions and the centroids GeoDataFrame
    return clipped_gdf, centroid_gdf

# Load contour and coastline polygons from the input files
jam_contour = gpd.read_file(input_file, layer="jam") 
# jamaica_convex_hull = gpd.read_file(input_file, layer="jamaica_convex")

# Create the Voronoi polygons clipped to Jamaica's convex hull
voronoi_polygons, voronoi_centroid_points = create_voronoi(
    jamaica_polygon = jamaica_convex_hull, 
    jam_contour = jam_contour,
    coastline_polygons = s_cape_flood_areas,
    intersections= scape_intersection,
    length_limit = 5_000,
    coastline= full_coastline_layer
)

# Save the Voronoi polygons and centroid points to a GeoPackage
layer_name = "STEP_5a_voronoi_polygons"
add_Layer_to_File(voronoi_polygons, processing_file, layer_name, "GPKG")

layer_name = "STEP_5a_voronoi_seed_pts"
add_Layer_to_File(voronoi_centroid_points, processing_file, layer_name, "GPKG")


Added 2 new points with c_ids [235, 236] for polygon 235
Added 2 new points with c_ids [237, 238] for polygon 226
Added 2 new points with c_ids [239, 240] for polygon 100
Removed old points for 235 redoing  num_pts
Added 3 new points with c_ids [241, 242, 243] for polygon 235
Added 2 new points with c_ids [244, 245] for polygon 231
----- Correcting edge case scenario for 222 -----
Added 2 new points with c_ids [246, 247] for polygon 222
Added 2 new points with c_ids [248, 249] for polygon 228
Added 2 new points with c_ids [250, 251] for polygon 233
----- Correcting edge case scenario for 221 -----
Added 2 new points with c_ids [252, 253] for polygon 221
Added 2 new points with c_ids [254, 255] for polygon 214
Added 2 new points with c_ids [256, 257] for polygon 230
----- Correcting edge case scenario for 232 -----
Added 2 new points with c_ids [258, 259] for polygon 232
----- Correcting edge case scenario for 83 -----
Added 2 new points with c_ids [260, 261] for polygon 83
Added 2 new 

In [155]:
from shapely.ops import linemerge

# Adapted and modified from https://github.com/thomas-fred/jam-coastal-protection

def combine_floods_within_voronoi(voronoi_polygons: gpd.GeoDataFrame,flood_polygons: gpd.GeoDataFrame,voronoi_points: gpd.GeoDataFrame,pts_id_field:str ,max_distance: float) -> gpd.GeoDataFrame:
    """
    Combines all parts of flood polygons that intersect with each Voronoi polygon,
    but remove any parts whose centroids are farther than a given distance from
    the corresponding Voronoi point.

    Args:
        voronoi_polygons (gpd.GeoDataFrame): GeoDataFrame containing the Voronoi polygons.
        flood_polygons (gpd.GeoDataFrame): GeoDataFrame containing the flood polygons to combine.
        voronoi_points (gpd.GeoDataFrame): GeoDataFrame containing the Voronoi points with a 'fid' column that matches the 'centroid_id' column.
        max_distance (float): Maximum allowed distance (in the same CRS units) between a polygon's centroid and the corresponding Voronoi point.

    Returns:
        gpd.GeoDataFrame: A GeoDataFrame containing the combined flood polygons for each Voronoi polygon,
                           with the flood areas merged based on their proximity to the Voronoi points.
    """
    combined_polygons = []  # List to store resulting geometries
    voronoi_ids = []  # List to store Voronoi IDs (optional, for tracking)

    # Loop through each Voronoi polygon
    for voronoi_index, voronoi in voronoi_polygons.iterrows():
        voronoi_geom = voronoi.geometry
        centroid_id = voronoi["centroid_id"]  # Get the Voronoi polygon's centroid ID

        # Get the corresponding Voronoi point based on the centroid ID
        voronoi_point = voronoi_points[voronoi_points[pts_id_field] == centroid_id]
        if voronoi_point.empty:
            continue  # Skip if no matching Voronoi point is found

        voronoi_point_geom = voronoi_point.geometry.iloc[0]

        # Find all flood polygons that intersect with the current Voronoi polygon
        intersecting_floods = flood_polygons[flood_polygons.intersects(voronoi_geom)]

        if not intersecting_floods.empty:
            # Combine the intersecting flood polygons within the Voronoi polygon
            combined_geom = intersecting_floods.intersection(voronoi_geom).union_all()

            # Filter out parts of the combined geometry that are too far from the Voronoi point
            if isinstance(combined_geom, MultiPolygon):
                # If the combined geometry is a MultiPolygon, check each part
                filtered_parts = [
                    part for part in combined_geom.geoms
                    if part.centroid.distance(voronoi_point_geom) <= max_distance
                ]
                combined_geom = MultiPolygon(filtered_parts) if filtered_parts else None
            elif combined_geom.centroid.distance(voronoi_point_geom) > max_distance:
                # If the combined geometry is a single polygon, check its centroid distance
                combined_geom = None

            if combined_geom:
                # Append the filtered geometry and its Voronoi index
                combined_polygons.append(combined_geom)
                voronoi_ids.append(voronoi_index)

    # Create a GeoDataFrame for the combined polygons
    result_gdf = gpd.GeoDataFrame({
        "id": voronoi_ids,  # Include the Voronoi IDs
        "geometry": combined_polygons  # Include the geometries
    }, crs=voronoi_polygons.crs)  # Set CRS to match Voronoi polygons

    return result_gdf  # Return the combined flood polygons GeoDataFrame


def linestring_intersect_polygons(default_inland_distance: float, coast: gpd.GeoDataFrame, polygons: gpd.GeoDataFrame, flood_polygons: gpd.GeoDataFrame) -> (gpd.GeoDataFrame, gpd.GeoDataFrame):
    """
    Sections the coastline based on intersections with each Voronoi polygon and individually 
    buffers the new linestring segments until they completely enclose the flood polygon 
    within the corresponding Voronoi section.

    Also returns a GeoDataFrame of intersections between the linestring and Voronoi polygons before any buffering.

    Args:
        default_inland_distance (float): Buffer radius in kilometers.
        coast (gpd.GeoDataFrame): GeoDataFrame containing the coastline geometry to be buffered.
        polygons (gpd.GeoDataFrame): GeoDataFrame of Voronoi polygons to check intersections.
        flood_polygons (gpd.GeoDataFrame): GeoDataFrame of flood polygons with a 'voronoi_id' column to associate with Voronoi polygons.

    Returns:
        tuple: 
            - GeoDataFrame containing the intersections of the linestring and Voronoi polygons before buffering.
            - GeoDataFrame containing the final intersected and buffered geometries, with Voronoi IDs.
    """
    # Prepare lists to store intersections and final buffered geometries
    initial_intersections = []
    final_intersections = []

    # Create a GeoDataFrame containing the union of all coastline geometries
    linestring = gpd.GeoDataFrame({"id": [0], "geometry": [coast.geometry.union_all()]})
    linestring.crs = coast.crs  # Set CRS to match the coast's CRS

    # Iterate over each Voronoi polygon
    for idx, poly in polygons.iterrows():
        # Check if the linestring intersects with the Voronoi polygon
        if linestring.geometry.iloc[0].intersects(poly.geometry):
            # Perform the intersection between the linestring and the Voronoi polygon
            intersection = linestring.geometry.iloc[0].intersection(poly.geometry)
            # print(poly['id'])
            
            # Append the raw intersection with the associated Voronoi ID
            initial_intersections.append({
                "geometry": intersection,
                "id": poly['id']
            })

            # Buffer the intersection piece separately based on the provided inland distance
            buffered_intersection = intersection.buffer(default_inland_distance * 1_000)
            
            # Find the corresponding flood polygon that has the same 'id'
            flood_polygon = flood_polygons[flood_polygons['id'] == idx]
            
            if not flood_polygon.empty:
                flood_geometry = flood_polygon.geometry.iloc[0]
                
                # Start iteratively buffering the intersection from 0.5 km and increase by 0.1 km
                current_buffer_radius = default_inland_distance * 1_000  # Start with a buffer radius of 0.1 km
                while not buffered_intersection.contains(flood_geometry):
                    # Increase buffer by 0.1 km at each iteration until it contains the flood geometry
                    current_buffer_radius += 0.1 * 1_000
                    buffered_intersection = intersection.buffer(current_buffer_radius)
            
            # Perform the final intersection with the flood polygon
            final_intersection = buffered_intersection.intersection(poly.geometry)
            
            # Append the final intersection with its associated Voronoi ID to the list
            final_intersections.append({
                "geometry": final_intersection,
                "id": poly['id']
            })
    
    # Create GeoDataFrames for initial and final intersections
    initial_intersections_gdf = gpd.GeoDataFrame(initial_intersections, crs=polygons.crs)
    final_intersections_gdf = gpd.GeoDataFrame(final_intersections, crs=polygons.crs)

    return initial_intersections_gdf, final_intersections_gdf


# Combine flood polygons for each Voronoi polygon using the function above
dbscan_voronoi_intersection = combine_floods_within_voronoi(
    voronoi_polygons = voronoi_polygons,  # Voronoi polygons GeoDataFrame
    flood_polygons = dbscan_flood_areas,  # Flood polygons GeoDataFrame
    voronoi_points = voronoi_centroid_points,  # Voronoi centroids GeoDataFrame
    pts_id_field= "c_id",
    max_distance = 8_000_000  # Maximum distance (in km) for filtering based on proximity to Voronoi points
)

# Add the resulting dbscan_voronoi_intersection layer to a processing file (GeoPackage format)
layer_name = "STEP_6_voronoi_flood_intersection"
add_Layer_to_File(dbscan_voronoi_intersection, processing_file, layer_name, "GPKG")

# Calculate initial and final intersections
flood_protection_coastline, flood_protection_areas = linestring_intersect_polygons(
    default_inland_distance=0.5,  # Buffer radius in kilometers
    coast=coastline_edge_layer,  # Coastline GeoDataFrame
    polygons=voronoi_polygons,  # Voronoi polygons GeoDataFrame
    flood_polygons=dbscan_voronoi_intersection,  # Flood polygons GeoDataFrame from previous step
)

flood_protection_coastline["geometry"] = flood_protection_coastline["geometry"].apply(lambda geom: linemerge(geom) if geom.geom_type == "MultiLineString" else geom)
flood_protection_areas = gpd.clip(flood_protection_areas, jamaica_polygon_revised)

# Save the raw intersections to the processing file
# add_Layer_to_File(flood_protection_coastline, processing_file, "flood_protection_coast", "GPKG")

# Save the final flood protection areas to the processing file
add_Layer_to_File(flood_protection_areas, processing_file, "STEP_7_flood_protection_areas", "GPKG")

In [143]:
def split_multipart_polygons(gdf, full_coastline_layer, length_limit=5000):
    new_features = []
    separated_parts = []  # Track only the separated small parts
    small_sections = []
    
    for _, row in gdf.iterrows():
        geom = row.geometry
        if isinstance(geom, MultiPolygon):  # Check if it's a multi-part polygon
            parts = list(geom.geoms)
            largest_part = max(parts, key=lambda p: p.area)  # Identify the largest part
            for part in parts:
                new_row = row.copy()
                new_row.geometry = part
                separated_parts.append(new_row)
                if part != largest_part:  # Ignore the largest part
                    new_row = row.copy()
                    new_row.geometry = part
                    small_sections.append(new_row)
        else:
            new_features.append(row)
    
    new_gdf = gpd.GeoDataFrame(new_features + separated_parts, columns=gdf.columns, crs=gdf.crs)
    new_gdf["id"] = range(1, len(new_gdf) + 1)  # Reassign IDs sequentially
    
    # Use a more robust method to identify small sections
    small_sections = new_gdf[new_gdf.geometry.apply(lambda geom: any(geom.equals(part.geometry) for part in small_sections))]
    add_Layer_to_File(small_sections, processing_file, "STEP_8_debug_split", "GPKG")
    touching_pairs = set()
    
    # Identify touching polygons and store as ordered pairs
    for _, poly in small_sections.iterrows():
        for _, other_poly in new_gdf.iterrows():
            if poly["id"] != other_poly["id"] and poly.geometry.buffer(0.5).intersects(other_poly.geometry.buffer(0.5)):
                pair = (poly["id"], other_poly["id"]) \
                    if poly.geometry.area < other_poly.geometry.area else \
                    (other_poly["id"], poly["id"])
                touching_pairs.add(pair)
    
    touching_pairs = list(touching_pairs)
    area_dict = {row["id"]: row.geometry.area for _, row in new_gdf.iterrows()}
    
    # Sort touching pairs by polygon area
    touching_pairs.sort(key=lambda x: area_dict[x[0]])

    buffer_distance = 0.5
    sorted_pairs = []

    best_pairs = {}  # Dictionary to store the best (first_id, second_id) pair

    for first_id, second_id in touching_pairs:
        first_polygon = new_gdf.loc[new_gdf['id'] == first_id, 'geometry'].values[0]
        second_polygon = new_gdf.loc[new_gdf['id'] == second_id, 'geometry'].values[0]
        
        shared_border_length = first_polygon.buffer(buffer_distance).intersection(second_polygon).length  

        # If first_id is not in best_pairs OR this pair has a longer shared border, update it
        if first_id not in best_pairs or shared_border_length > best_pairs[first_id][1]:
            best_pairs[first_id] = (second_id, shared_border_length)

    # Convert dictionary back to a list of tuples
    sorted_pairs = [(first_id, second_id) for first_id, (second_id, _) in best_pairs.items()]

    merged_gdf = new_gdf.copy()
    srtdp = sorted_pairs.copy()
    merged_polygons = []
    
    i = 0
    while i < len(srtdp):
        first_id, second_id = srtdp[i]
        
        poly1 = merged_gdf.loc[merged_gdf["id"] == first_id, "geometry"].values[0]
        poly2 = merged_gdf.loc[merged_gdf["id"] == second_id, "geometry"].values[0]
        
        # Check if the merged polygon would exceed the coastline length limit
        potential_merged_polygon = poly1.union(poly2.buffer(buffer_distance))
        coastline_length = get_coastline_length(potential_merged_polygon, full_coastline_layer)
        
        if coastline_length > length_limit:
            # Skip this merge and move to the next pair
            i += 1
            continue
        
        new_polygon = potential_merged_polygon  # Use the already computed merged polygon
        new_id = first_id
        
        merged_gdf.loc[merged_gdf["id"] == first_id, "geometry"] = new_polygon
        merged_polygons.append(second_id)
        
        srtdp = [(new_id if x == second_id else x, new_id if y == second_id else y) for x, y in srtdp]
        i += 1
    
    merged_gdf = merged_gdf[~merged_gdf['id'].isin(merged_polygons)].reset_index(drop=True)
    add_Layer_to_File(merged_gdf, processing_file, "STEP_8_debug_split_2", "GPKG")

    return merged_gdf

# Function to get coastline length
def get_coastline_length(geom, full_coastline_layer):
    # Buffer the geometry slightly to ensure intersection with coastline
    buffered_geom = geom.buffer(10)
    
    # Find intersection with coastline
    coast_intersection = full_coastline_layer.geometry.iloc[0].intersection(buffered_geom)
    
    # Calculate length of the intersection
    if coast_intersection.is_empty:
        return 0
    elif hasattr(coast_intersection, 'geoms'):  # MultiLineString
        return sum(line.length for line in coast_intersection.geoms)
    else:  # Single LineString
        return coast_intersection.length

# Usage
flood_protection_areas_v2 = split_multipart_polygons(flood_protection_areas, full_coastline_layer, length_limit=5000)
add_Layer_to_File(flood_protection_areas_v2, processing_file, "STEP_8_flood_protection_areas_v2", "GPKG")

In [144]:
def merge_adjacent_polygons_by_coastline(gdf, coastline, length_limit):
    touching_pairs = set()
    
    # Identify touching polygons and store as ordered pairs
    for _, poly in gdf.iterrows():
        for _, other_poly in gdf.iterrows():
            if poly["id"] != other_poly["id"] and poly.geometry.buffer(0.5).intersects(other_poly.geometry.buffer(0.5)):
                pair = (poly["id"], other_poly["id"]) \
                    if poly.geometry.area < other_poly.geometry.area else \
                    (other_poly["id"], poly["id"])
                touching_pairs.add(pair)
    
    touching_pairs = list(touching_pairs)
    max_area = np.mean(gdf.geometry.area.values) / 2
    area_dict = {row["id"]: row.geometry.area for _, row in gdf.iterrows()}
    
    # Sort touching pairs by polygon area
    touching_pairs.sort(key=lambda x: area_dict[x[0]])
    
    buffer_distance = 0.5
    sorted_pairs = []
    
    for first_id, second_id in touching_pairs:
        if area_dict[first_id] > max_area:
            continue  
        
        first_polygon = gdf.loc[gdf['id'] == first_id, 'geometry'].values[0]
        second_polygon = gdf.loc[gdf['id'] == second_id, 'geometry'].values[0]
        shared_border_length = first_polygon.buffer(buffer_distance).intersection(second_polygon).length  
        length = first_polygon.buffer(buffer_distance).intersection(coastline).length.iloc[0] + second_polygon.buffer(buffer_distance).intersection(coastline).length.iloc[0]
        
        if length <= length_limit:
            sorted_pairs.append((first_id, second_id, shared_border_length))
    
    sorted_pairs.sort(key=lambda x: (area_dict[x[0]], -x[2]))  
    sorted_pairs = [(x[0], x[1]) for x in sorted_pairs]
    
    merged_gdf = gdf.copy()
    srtdp = sorted_pairs.copy()
    merged_polygons = []
    
    i = 0
    while i < len(srtdp):
        first_id, second_id = srtdp[i]
        
        poly1 = merged_gdf.loc[merged_gdf["id"] == first_id, "geometry"].values[0]
        poly2 = merged_gdf.loc[merged_gdf["id"] == second_id, "geometry"].values[0]
        length1 = poly1.buffer(buffer_distance).intersection(coastline).length.iloc[0]
        length2 = poly2.buffer(buffer_distance).intersection(coastline).length.iloc[0]
        
        new_coastline_length = length1 + length2
        
        if new_coastline_length < length_limit:
            new_polygon = poly1.union(poly2.buffer(buffer_distance))
            new_coastline_intersection = new_polygon.buffer(buffer_distance).intersection(coastline).length.iloc[0]
            new_id = first_id
            
            merged_gdf.loc[merged_gdf["id"] == first_id, "geometry"] = new_polygon
            # merged_gdf.loc[merged_gdf["id"] == first_id, "coastline_length"] = new_coastline_intersection
            merged_polygons.append(second_id)
            
            srtdp = [(new_id if x == second_id else x, new_id if y == second_id else y) for x, y in srtdp]
        
        i += 1
    
    merged_gdf = merged_gdf[~merged_gdf['id'].isin(merged_polygons)].reset_index(drop=True)
    return merged_gdf
    

length_limit = 5_000
flood_protection_areas_v3 = merge_adjacent_polygons_by_coastline(flood_protection_areas_v2, full_coastline_layer, length_limit)
add_Layer_to_File(flood_protection_areas_v3, processing_file, "STEP_9_flood_protection_areas_v3", "GPKG")


In [145]:
max_area = np.mean(flood_protection_areas_v2.geometry.area.values)/2 
print (max_area)

1014378.6250310937


In [146]:
final_coastal_protection_area = add_max_zonal_stats(
        geopkg = flood_protection_areas_v3,  # The cleaned GeoDataFrame
        raster_path = raster_file,  # Path to the raster file containing flood height data
        new_column_name= "max_flood_height"  # New column to store the maximum flood height value
    )

layer_name = "STEP_10_final_protection_area"
add_Layer_to_File(final_coastal_protection_area, processing_file, layer_name, "GPKG")

In [147]:
def find_coast_segment_for_polygon(coastline, polygon):
    # if coastline.geometry.iloc[0].intersects(polygon.geometry):
    intersection = coastline.geometry.iloc[0].intersection(polygon.geometry.buffer(0.1))
    return intersection

flood_protection_coastline = []

for idx, poly in final_coastal_protection_area.iterrows():
    intersection = find_coast_segment_for_polygon(full_coastline_layer, poly)
    # print (intersection.length)
    flood_protection_coastline.append({
        "geometry": intersection,
        "id": poly["id"],
        "length": intersection.length
    })

flood_protection_coastline = gpd.GeoDataFrame(flood_protection_coastline, crs=flood_protection_areas_v2.crs)
flood_protection_coastline = flood_protection_coastline.merge(final_coastal_protection_area[['id', 'max_flood_height']], on='id', how='left')

add_Layer_to_File(flood_protection_coastline, processing_file, "STEP_11_final_flood_protection_coast", "GPKG")